# 02 · The specialist agents: tools, RAG, MCP and ReAct

**You'll learn:** what an agent's tools are, how the fraud-case search (RAG) and the live MCP geolocation server work, how a ReAct agent decides which tool to call, how the three specialists differ, and how the aggregator turns their findings into one score.

**Time:** about 10 minutes · **Cost:** free tier or about 1¢ · **Needs:** an API key in `.env` (`GEMINI_API_KEY` by default).

```mermaid
flowchart LR
  T["borderline transaction"] --> TA["transaction_analyst<br/>account_history · merchant_risk_score · sanctions_check"]
  T --> IV["identity_verifier<br/>device_fingerprint · geolocation (live MCP)"]
  T --> PD["pattern_detector<br/>account_history · similar_fraud_cases (RAG)"]
  TA --> AG["aggregator<br/>structured output: composite_score + reasoning"]
  IV --> AG
  PD --> AG
```

| Part | What it covers | Code |
|---|---|---|
| 1 | Setup | |
| 2 | Tools: what agents can call | `tools/llm_tools.py`, `tools/mock_apis.py` |
| 3 | RAG: searching past fraud cases | `tools/mock_apis.py: search_similar_fraud_cases()` |
| 4 | MCP: a live geolocation server | `mcp_server/geolocation_server.py`, `tools/mcp_geolocation.py` |
| 5 | ReAct: one agent, step by step | `core/specialist_factory.py` |
| 6 | All three specialists | `agents/` |
| 7 | The aggregator | `nodes/aggregator.py` |
| 8 | Recap and things to try | |

---
## Part 1 · Setup

### Setup
The next cell connects the notebook to the project. It gives this notebook **its own** database files in `.notebook_state/`, so nothing here touches the Streamlit app's data. It prints which LLM this run uses and whether its key is set (never the key itself). Set `SHOW_LIVE_LOGS = True` to see each node's live log line.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
SHOW_LIVE_LOGS = False  # True = print every node's live log line, like the terminal does

# Setup: run from the repo root, and give this notebook its own database files in
# .notebook_state/ so it never touches the app's checkpoints.sqlite / account_memory.sqlite.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
STATE_DIR = ROOT / ".notebook_state"
STATE_DIR.mkdir(exist_ok=True)
os.environ["CHECKPOINT_DB_PATH"] = str(STATE_DIR / "02_checkpoints.sqlite")
os.environ["MEMORY_DB_PATH"] = str(STATE_DIR / "02_memory.sqlite")
os.environ.setdefault("FASTMCP_LOG_LEVEL", "WARNING")  # quiet the MCP server subprocess
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # never overrides the two database paths set above

import pandas as pd

import core.audit  # noqa: F401  (creates the "fraud_guard" live logger)
from core.llm import llm_model, llm_provider

logging.getLogger("fraud_guard").setLevel(logging.INFO if SHOW_LIVE_LOGS else logging.WARNING)
pd.set_option("display.max_colwidth", 160)
key = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}.get(llm_provider(), "?")
print(f"LLM: {llm_provider()} / {llm_model()}   ({key} set: {'yes' if os.environ.get(key) else 'NO'})")

---
## Part 2 · Tools: what agents can call

A **tool** is a Python function the LLM may ask to run. The `@tool` decorator turns its name, docstring and arguments into a description the model reads to decide when to call it. The model never runs code itself: it replies "call `account_history` with `acct_2002`", the framework runs the function, and the result goes back to the model.

In [ ]:
from tools.llm_tools import account_history, device_fingerprint, merchant_risk_score, sanctions_check, similar_fraud_cases

pd.DataFrame([
    {"tool": t.name, "arguments": list(t.args), "description the LLM reads": t.description}
    for t in (account_history, merchant_risk_score, sanctions_check, device_fingerprint, similar_fraud_cases)
])

Calling tools directly, the way an agent would, for the two wire transfers. Note the sanctions check: `borderline_restart_demo` pays "Silk Road Holdings Ltd", which is on the mock watchlist.

In [ ]:
from data.synthetic_transactions import SCENARIOS

for name in ("borderline_wire", "borderline_restart_demo"):
    t = SCENARIOS[name]
    print(f"===== {name}: ${t['amount']:,.0f} to {t['beneficiary_name']}")
    print("account_history     ", account_history.invoke({"account_id": t["account_id"]}))
    print("merchant_risk_score ", merchant_risk_score.invoke({"merchant_id": t["merchant_id"]}))
    print("device_fingerprint  ", device_fingerprint.invoke({"device_id": t["device_id"]}))
    print("sanctions_check     ", sanctions_check.invoke({"beneficiary_name": t["beneficiary_name"]}), "\n")

---
## Part 3 · RAG: searching past fraud cases

`similar_fraud_cases` is retrieval-augmented generation in miniature: a handful of past confirmed fraud cases are embedded into an in-memory **Chroma** vector store (using the embedding model of your provider), and the tool returns the 2 most similar to a description. The `pattern_detector` uses it to ask "have we seen this before?".

In [ ]:
from tools.mock_apis import FRAUD_CASE_TEXTS

print(f"{len(FRAUD_CASE_TEXTS)} past cases in the store, e.g.:\n- {FRAUD_CASE_TEXTS[0]}\n")
query = "Large international wire transfer to a brand-new beneficiary from a business account"
for case in similar_fraud_cases.invoke({"description": query}):
    print("match:", case)

---
## Part 4 · MCP: a live geolocation server

Everything else in the demo is mock data, but the `identity_verifier`'s geolocation is **real**. It comes from a separate **MCP server** (Model Context Protocol, a standard way to expose tools to AI apps):

```mermaid
flowchart LR
  A["identity_verifier agent"] --> C["MCP client<br/>langchain-mcp-adapters"]
  C -->|"stdio: starts a subprocess"| S["mcp_server/geolocation_server.py<br/>FastMCP tool: geolocation(ip)"]
  S -->|"HTTP"| API["ip-api.com"]
```

The first call starts the server process; `get_geolocation_tool()` wraps its async tool for the synchronous agents. Compare the live answer with the mock table triage used:

In [ ]:
from tools.mcp_geolocation import get_geolocation_tool
from tools.mock_apis import get_geolocation

geo = get_geolocation_tool()
for name in ("impossible_travel", "borderline_wire"):
    t = SCENARIOS[name]
    print(f"{name}: mock {t['ip']} -> {get_geolocation(t['ip'])} | live {t['real_ip']} -> {geo.invoke({'ip': t['real_ip']})}")

**What to notice:** the impossible-travel IP genuinely resolves to Nigeria, which is what makes the agent's write-up convincing rather than scripted.

---
## Part 5 · ReAct: one agent, step by step

Each specialist is a **ReAct agent** (reason + act), built with LangGraph's `create_react_agent`:

```mermaid
flowchart LR
  P["system prompt + transaction"] --> L{"LLM"}
  L -->|"tool call"| T["run the tool"]
  T -->|"result"| L
  L -->|"no more tool calls"| F["final finding: 2–3 sentences"]
```

The LLM keeps choosing tools until it has enough, then writes its finding. `make_specialist_node()` wraps an agent into a graph node and records every tool call and result in the audit log. Running the transaction analyst on the wire to "Silk Road Holdings Ltd":

In [ ]:
from agents.transaction_analyst import SYSTEM_PROMPT, transaction_analyst_node
from core.state import new_state

print("SYSTEM PROMPT:", SYSTEM_PROMPT, "\n")
update = transaction_analyst_node(new_state(SCENARIOS["borderline_restart_demo"]))

trace = pd.DataFrame(update["audit_log"])
display(trace[[c for c in ("kind", "tool", "args", "result", "text") if c in trace]])
print("FINDING:", update["analyst_findings"])

**What to notice:** the model decided on its own to call `sanctions_check` because the prompt says to when a beneficiary is present, and it leads its finding with the watchlist hit. Adding that skill took one more tool in a list; the graph did not change.

---
## Part 6 · All three specialists

Same factory, different prompts and tools. Each investigates one independent angle, which is why they can run in parallel in the graph. Here they run one after another on the $18,000 wire:

In [ ]:
from agents.identity_verifier import identity_verifier_node
from agents.pattern_detector import pattern_detector_node

state = new_state(SCENARIOS["borderline_wire"])
findings = {}
for node, key in ((transaction_analyst_node, "analyst_findings"),
                  (identity_verifier_node, "identity_findings"),
                  (pattern_detector_node, "pattern_findings")):
    result = node(state)
    findings[key] = result[key]
    tools_used = [e["tool"] for e in result["audit_log"] if e["kind"] == "tool_call"]
    print(f"===== {key} (tools: {tools_used})\n{result[key]}\n")

---
## Part 7 · The aggregator

Three free-text opinions must become **one** number a router can compare. The aggregator asks the LLM for **structured output**: a Pydantic model with exactly two fields, so the reply is always parseable.

In [ ]:
from nodes.aggregator import AGGREGATOR_THRESHOLD, AggregatorOutput, aggregator_node, route_after_aggregator
from nodes.triage import triage_node

print({name: field.description for name, field in AggregatorOutput.model_fields.items()}, "\n")
state = {**state, **triage_node(state), **findings}
result = aggregator_node(state)
print(f"composite_score = {result['composite_score']:.2f}  (human review at ≥ {AGGREGATOR_THRESHOLD})")
print("reasoning:", result["aggregate_reasoning"])
print("next node:", route_after_aggregator({**state, **result}))

---
## Part 8 · Recap and things to try

**What happened:** tools gave the agents facts (mock services, a vector search over past fraud, and a live MCP server); each ReAct agent chose its own tool calls and wrote a short finding; the aggregator merged them into a structured score that decides whether a human must look.

**Try this**
- Run `identity_verifier_node` on `SCENARIOS["impossible_travel"]` and read how it uses the live Nigeria lookup.
- Search the fraud cases for "small test charges followed by a large purchase".
- Add a tool: write a `@tool` function and add it to one specialist's list in `agents/`. Nothing else changes.

**Next:** `03_full_graph.ipynb` runs all of this inside the real graph, including the pause for human review.